In [1]:
import json, sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm


## Loading the environment

In [2]:
DB = Path("../data/hg19.db")
MUT = Path("../data/simple_breast.csv")

Mutations

In [3]:
mutations = pd.read_csv(MUT,
                        dtype={"sampleID":str,
                               "chr":str,
                               "pos":int,
                               "ref":str,
                               "mut":str}).iloc[:, :5]

In [4]:
mutations.head()

,sampleID,chr,pos,ref,mut
0,Sample_1,1,871244,G,C
1,Sample_1,1,6648841,C,G
2,Sample_1,1,17557072,G,A
3,Sample_1,1,22838492,G,C
4,Sample_1,1,27097733,G,A


In [5]:
mutations.shape

(18637, 5)

In [6]:
mutations.dropna().reset_index(drop=True, inplace=True)

In [7]:
mutations.shape

(18637, 5)

Exclude 0/0 mutations

In [8]:
mutations = mutations[mutations["ref"] != mutations["mut"]].reset_index(drop=True)

In [9]:
mutations.head()

,sampleID,chr,pos,ref,mut
0,Sample_1,1,871244,G,C
1,Sample_1,1,6648841,C,G
2,Sample_1,1,17557072,G,A
3,Sample_1,1,22838492,G,C
4,Sample_1,1,27097733,G,A


In [10]:
mutations.shape

(18637, 5)

Contiguous / Duplicate mutations

In [11]:
mutations[mutations.sort_values(
        ["sampleID", "chr", "pos"])
        .groupby("sampleID")["pos"].diff() == 1]

,sampleID,chr,pos,ref,mut


In [12]:
mutations[mutations[["chr", "pos", "ref", "mut"]].duplicated()].sort_values(["pos", "ref", "mut", "chr"])

,sampleID,chr,pos,ref,mut
13011,Sample_5,11,533874,T,C
7328,Sample_174,X,5811421,C,T
18157,Sample_91,17,7577022,G,A
16941,Sample_8,17,7577094,G,A
8685,Sample_187,17,7577539,G,A
...,...,...,...,...,...
14638,Sample_61,2,209113112,C,T
16893,Sample_79,2,209113112,C,T
18204,Sample_92,2,209113112,C,T
18579,Sample_98,2,209113112,C,T


## Load Reference Coding sequence

In [13]:
conn = sqlite3.connect(DB)

In [14]:
refcds = pd.read_sql("SELECT * FROM refcds", conn)

In [15]:
refcds.head()

,gene_name,gene_id,protein_id,CDS_length,chr,strand,intervals_splice,seq_cds,seq_cds1up,seq_cds1down,seq_splice,seq_splice1up,seq_splice1down,L
0,A1BG,ENSG00000121410,ENSP00000263100,1488,19,-1,"[58858396,58858397,58858714,58858717,58858718,...",ATGTCCATGCTCGTGGTCTTTCTCTTGCTGTGGGGTGTCACCTGGG...,CATGTCCATGCTCGTGGTCTTTCTCTTGCTGTGGGGTGTCACCTGG...,TGTCCATGCTCGTGGTCTTTCTCTTGCTGTGGGGTGTCACCTGGGG...,GACTGGAGTGGAGTGGACTGGAGTGGAGTGGAGTG,ATAGGACAGGACAGGACAGGACAGAACAGTACAGG,AGGGTGGCGTAGCGTCGTGTAGCGTTGTGTGGGGT,"[[0,7,0,0],[3,4,0,0],[0,7,0,0],[0,12,0,0],[1,1..."
1,A1CF,ENSG00000148584,ENSP00000363105,1785,10,-1,"[52566641,52566642,52569649,52569652,52569653,...",ATGGAATCAAATCACAAATCCGGGGATGGATTGAGCGGCACTCAGA...,AATGGAATCAAATCACAAATCCGGGGATGGATTGAGCGGCACTCAG...,TGGAATCAAATCACAAATCCGGGGATGGATTGAGCGGCACTCAGAA...,GAGTGGAGTGGAGTGGACTGGAGTGGAGTGGAGTGGAGTGGAGTGG...,ACTGGACCGTATGGGATAGGACAGGACGGGACAGGACAGGACAGAA...,GGGATCGTGTAGTATAGAATGGGATGGAATGGCTTAGCATAGTGTG...,"[[0,66,0,0],[22,44,0,0],[0,55,11,0],[0,24,0,0]..."
2,A2M,ENSG00000175899,ENSP00000323929,4425,12,-1,"[9220436,9220437,9220774,9220777,9220778,92208...",ATGGGGAAGAACAAACTCCTTCATCCAAGTCTGGTTCTTCTCCTCT...,CATGGGGAAGAACAAACTCCTTCATCCAAGTCTGGTTCTTCTCCTC...,TGGGGAAGAACAAACTCCTTCATCCAAGTCTGGTTCTTCTCCTCTT...,GAGTGGAGTGGAGTGGAGTGGAGTGGAGTGGAGTGGAGTGGAGTGG...,ACAGGACAGGACAGGACGGGACAGGATAGGACGGGAAAGGACGGTA...,AGCATAGTGTGGAATCGTATTGAATAGAGTGGTTTAGAGTCGTATG...,"[[0,89,0,0],[21,68,0,0],[0,74,15,0],[0,63,0,2]..."
3,A2ML1,ENSG00000166535,ENSP00000299698,4365,12,1,"[8975310,8975311,8975314,8975776,8975777,89759...",ATGTGGGCTCAGCTCCTTCTAGGAATGTTGGCCCTATCACCAGCCA...,GATGTGGGCTCAGCTCCTTCTAGGAATGTTGGCCCTATCACCAGCC...,TGTGGGCTCAGCTCCTTCTAGGAATGTTGGCCCTATCACCAGCCAT...,GTGAGGTGAGGTGAGGTGAGGTGAGGTGAGGTGAGGTGAGGTGAGG...,CGATATGACAGGACAGGACAGGACAGGGCAGGATAGGATAGGACAG...,TGTGATACGGTAAGTTGTGTTACGGTATGTTAAGGTGTGATATGGT...,"[[0,75,0,1],[17,58,0,1],[0,60,15,1],[0,60,0,1]..."
4,A3GALT2,ENSG00000184389,ENSP00000475261,1023,1,-1,"[33773055,33773056,33777648,33777651,33777652,...",ATGGCTCTCAAGGAGGGACTCAGGGCCTGGAAGAGAATCTTCTGGC...,TATGGCTCTCAAGGAGGGACTCAGGGCCTGGAAGAGAATCTTCTGG...,TGGCTCTCAAGGAGGGACTCAGGGCCTGGAAGAGAATCTTCTGGCG...,GAGTGGAGTGGAATGGAGTG,ATAGGACAGGACAGGACAGG,AGGATGGAATGGAATGGGGT,"[[0,1,0,1],[0,1,0,1],[0,1,0,1],[0,6,0,0],[1,5,..."


In [16]:
# intervals = pd.read_sql("SELECT gene_name, start, end FROM cds_intervals ORDER BY start", conn)

# intervals = pd.read_sql("SELECT gene_name, start, end FROM cds_intervals ORDER BY gene_name, rowid", conn)

# Sort by order of exonic regions
intervals = pd.read_sql("SELECT gene_name, start, end FROM cds_intervals ORDER BY gene_name, start", conn)

In [17]:
intervals.head()

,gene_name,start,end
0,A1BG,58858388,58858395
1,A1BG,58858719,58859006
2,A1BG,58861736,58862017
3,A1BG,58862757,58863053
4,A1BG,58863649,58863921


In [18]:
gene_cds = {name: list(zip(df["start"], df["end"])) for name, df in intervals.groupby("gene_name")}

In [19]:
all(type(row["CDS_length"]) == int for _, row in refcds[:2].iterrows())

True

In [20]:
all(type(row["strand"]) == int for _, row in refcds[:2].iterrows())

True

In [21]:
json.loads(refcds.iloc[0]["intervals_splice"])[:10]

[58858396,
 58858397,
 58858714,
 58858717,
 58858718,
 58859007,
 58859008,
 58861731,
 58861734,
 58861735]

In [22]:
RefCDS = {
    row["gene_name"]: {
        "gene_name": row["gene_name"],
        "gene_id": row["gene_id"],
        "protein_id": row["protein_id"],
        "CDS_length": int(row["CDS_length"]),
        "chr": str(row["chr"]),
        "strand": int(row["strand"]),
        "intervals_cds": gene_cds.get(row["gene_name"], []),
        "intervals_splice": set(json.loads(row["intervals_splice"])),
        "seq_cds": list(row["seq_cds"] or ""),
        "seq_cds1up": list(row["seq_cds1up"] or ""),
        "seq_cds1down": list(row["seq_cds1down"] or ""),
        "seq_splice": list(row["seq_splice"] or ""),
        "seq_splice1up": list(row["seq_splice1up"] or ""),
        "seq_splice1down": list(row["seq_splice1down"] or ""),
        "L": np.array(json.loads(row["L"]), dtype=np.int32),
        "N": np.zeros((192, 4), dtype=np.int32),
    }
    for _, row in refcds.iterrows()
}

In [23]:
RefCDS["BRCA1"].keys()

dict_keys(['gene_name', 'gene_id', 'protein_id', 'CDS_length', 'chr', 'strand', 'intervals_cds', 'intervals_splice', 'seq_cds', 'seq_cds1up', 'seq_cds1down', 'seq_splice', 'seq_splice1up', 'seq_splice1down', 'L', 'N'])

In [24]:
genes = list(RefCDS.keys())
gidx = {g: i for i, g in enumerate(genes)}

In [25]:
genes[0], gidx[genes[0]]

('A1BG', 0)

Load substitution model, covariates, known cancer genes

In [26]:
# Order needs to be consistent with the order of genes in RefCDS
subs = pd.read_sql("""
    SELECT syn, mis, non, spl
    FROM substmodel
    ORDER BY idx
    """, conn)

In [27]:
covs_df = pd.read_sql("SELECT * FROM covariates", conn).set_index("gene_name")

In [28]:
# Force conversion to avoid dtype Object in case of NULL values
covs = covs_df.reindex(refcds["gene_name"].tolist()).values.astype(float)


In [29]:
subs.head()

,syn,mis,non,spl
0,t*AAA>ACA,t*AAA>ACA*wmis,t*AAA>ACA*wnon,t*AAA>ACA*wspl
1,t*AAA>AGA,t*AAA>AGA*wmis,t*AAA>AGA*wnon,t*AAA>AGA*wspl
2,t*AAA>ATA,t*AAA>ATA*wmis,t*AAA>ATA*wnon,t*AAA>ATA*wspl
3,t*AAC>ACC,t*AAC>ACC*wmis,t*AAC>ACC*wnon,t*AAC>ACC*wspl
4,t*AAC>AGC,t*AAC>AGC*wmis,t*AAC>AGC*wnon,t*AAC>AGC*wspl


In [30]:
covs

array([[-1.27962801e+01,  5.59980658e+00, -8.10038078e+00, ...,
         1.03334192e+00, -8.47727163e-01, -6.60590046e-01],
       [ 7.86580763e+00,  8.78632739e+00,  2.59035210e+00, ...,
        -4.04755980e-01,  5.12074592e-01, -3.16150236e-01],
       [ 4.78366186e+00,  1.72140588e+00,  5.54362878e+00, ...,
         7.49726357e-01,  7.66669581e-01, -1.73169997e+00],
       ...,
       [-2.04090278e+01, -8.62929481e+00, -6.21330431e+00, ...,
        -7.98661917e-02, -7.50204348e-01,  1.71619528e-01],
       [-3.90112798e+00, -8.67520773e+00,  6.12438945e+00, ...,
         6.67127414e-03, -1.25717371e-01,  5.60698914e-01],
       [ 2.29976585e+00, -7.20193905e+00,  8.13979240e+00, ...,
         1.58776858e+00, -3.16655013e-01,  3.03470466e-01]],
      shape=(20091, 20))

In [31]:
cancer_genes = set(pd.read_sql("SELECT gene_name FROM known_cancer_genes", conn)["gene_name"])

In [32]:
list(cancer_genes)[:10]

['IKZF1',
 'SDHA',
 'MED12',
 'LEF1',
 'CHD4',
 'ZNF384',
 'MYCL',
 'CRTC3',
 'RALGDS',
 'ERCC5']

## Annotations

In [33]:
nt = ["A", "C", "G", "T"]

Trinucleotides are all $4*4*4 = 64$ of codon combinations

In [34]:
tri = [a + b + c for a in nt for b in nt for c in nt]

In [35]:
print(tri[:3], len(tri))

['AAA', 'AAC', 'AAG'] 64


In [36]:
tridx = {sub: i for i, sub in enumerate(tri)}

In [37]:
list(tridx.items())[:3]

[('AAA', 0), ('AAC', 1), ('AAG', 2)]

Substitution model is {flankin base} {mutated} {flank}

ABC>ADC (D not B)

64*3 = 192 mutations

In [38]:
trimsubs = [f"{t}>{t[0]}{alt}{t[2]}" for t in tri for alt in nt if alt != t[1]]

In [39]:
trimsubs[:3], len(trimsubs), len(set(trimsubs))

(['AAA>ACA', 'AAA>AGA', 'AAA>ATA'], 192, 192)

In [40]:
trimsubsidx = {sub: i for i, sub in enumerate(trimsubs)}

In [41]:
revc = str.maketrans("ACGT", "TGCA")

In [42]:
subsmodel = subs.values

In [43]:
subsmodel[:3], subsmodel.shape

(array([['t*AAA>ACA', 't*AAA>ACA*wmis', 't*AAA>ACA*wnon',
         't*AAA>ACA*wspl'],
        ['t*AAA>AGA', 't*AAA>AGA*wmis', 't*AAA>AGA*wnon',
         't*AAA>AGA*wspl'],
        ['t*AAA>ATA', 't*AAA>ATA*wmis', 't*AAA>ATA*wnon',
         't*AAA>ATA*wspl']], dtype=object),
 (192, 4))

Fix vcf for deletions

Original VCF

pos,ref,mut  
100,CA,C

Corrected

101,CA,

As VCF's don't allow empty fields

In [44]:
mutations["start"] = mutations["pos"]

In [45]:
mutations["start"]

0           871244
1          6648841
2         17557072
3         22838492
4         27097733
           ...    
18632     43029309
18633    151939113
18634     77764723
18635    108264124
18636     71792551
Name: start, Length: 18637, dtype: int64

In [46]:
mutations["end"] = mutations["pos"] + mutations["ref"].str.len() - 1

In [47]:
mutations.loc[0:3, ["start", "end"]]

,start,end
0,871244,871244
1,6648841,6648841
2,17557072,17557072
3,22838492,22838492


In [48]:
_del_anchor = (
    (mutations["ref"].str[0] == mutations["mut"].str[0]) &
    (mutations["ref"].str.len() > mutations["mut"].str.len())
)

In [49]:
_del_anchor.sum()

np.int64(0)

In [50]:
mutations.loc[_del_anchor, "start"] += 1

In [51]:
intervals = pd.read_sql("""
    SELECT gene_name, chr,
        start AS interval_start, end AS interval_end
    FROM cds_intervals
    """, conn)

Keep only mutations that are in one of the CDS regions

In [52]:
intervals

,gene_name,chr,interval_start,interval_end
0,A1BG,19,58858388,58858395
1,A1BG,19,58858719,58859006
2,A1BG,19,58861736,58862017
3,A1BG,19,58862757,58863053
4,A1BG,19,58863649,58863921
...,...,...,...,...
194922,ZZZ3,1,78046683,78046754
194923,ZZZ3,1,78047461,78047576
194924,ZZZ3,1,78047664,78047811
194925,ZZZ3,1,78050202,78050340


In [53]:
mut_idx = mutations.reset_index(names="mut_idx")

In [54]:
mut_idx

,mut_idx,sampleID,chr,pos,ref,mut,start,end
0,0,Sample_1,1,871244,G,C,871244,871244
1,1,Sample_1,1,6648841,C,G,6648841,6648841
2,2,Sample_1,1,17557072,G,A,17557072,17557072
3,3,Sample_1,1,22838492,G,C,22838492,22838492
4,4,Sample_1,1,27097733,G,A,27097733,27097733
...,...,...,...,...,...,...,...,...
18632,18632,Sample_99,6,43029309,A,C,43029309,43029309
18633,18633,Sample_99,6,151939113,T,C,151939113,151939113
18634,18634,Sample_99,8,77764723,A,G,77764723,77764723
18635,18635,Sample_99,8,108264124,A,G,108264124,108264124


In [59]:
mutations["start"].values[:, None]

array([[   871244],
       [  6648841],
       [ 17557072],
       ...,
       [ 77764723],
       [108264124],
       [ 71792551]], shape=(18637, 1))

In [75]:
intervals["interval_start"].values[None, :]

array([[58858388, 58858719, 58861736, ..., 78047664, 78050202, 78097535]],
      shape=(1, 194927))

In [61]:
pairs = []

In [62]:
for chr, iv in intervals.groupby("chr"):
    mc = mutations[mutations["chr"]==chr]

    if mc.empty:
        continue

    idx = mc.index.values

    # [:, None] convert 1D array to 2D column vector for broadcasting
    ms = mc["start"].values[:, None]
    me = mc["end"].values[:, None]
    ivs = iv["interval_start"].values[None, :]
    ive = iv["interval_end"].values[None, :]

    gn = iv["gene_name"].values

    # Index of mutations and intervals that overlap
    mi, ii = np.where((ivs <= me) & (ive >= ms))

    pairs.extend(zip(idx[mi], gn[ii]))


In [63]:
pairs[:10], len(pairs)

([(np.int64(0), 'SAMD11'),
  (np.int64(1), 'ZBTB48'),
  (np.int64(2), 'PADI1'),
  (np.int64(3), 'ZBTB40'),
  (np.int64(4), 'ARID1A'),
  (np.int64(5), 'FAM46B'),
  (np.int64(6), 'NCDN'),
  (np.int64(7), 'CC2D1B'),
  (np.int64(8), 'JAK1'),
  (np.int64(9), 'HHLA3')],
 18408)

In [64]:
mut_gene = pd.DataFrame(pairs, columns=["mut_idx","gene_name"]).drop_duplicates()

In [65]:
mut_gene.head(), mut_gene.shape

(   mut_idx gene_name
 0        0    SAMD11
 1        1    ZBTB48
 2        2     PADI1
 3        3    ZBTB40
 4        4    ARID1A,
 (18408, 2))

In [66]:
mutations = mutations.iloc[mut_gene["mut_idx"]].reset_index(drop=True)

In [67]:
mutations.shape

(18408, 7)

In [68]:
mutations['gene'] = mut_gene["gene_name"].values

In [69]:
mutations['gene'].head()

0    SAMD11
1    ZBTB48
2     PADI1
3    ZBTB40
4    ARID1A
Name: gene, dtype: str

In [71]:
mutations = mutations.drop_duplicates().reset_index(drop=True)

In [72]:
mutations.head(), mutations.shape

(   sampleID chr       pos ref mut     start       end    gene
 0  Sample_1   1    871244   G   C    871244    871244  SAMD11
 1  Sample_1   1   6648841   C   G   6648841   6648841  ZBTB48
 2  Sample_1   1  17557072   G   A  17557072  17557072   PADI1
 3  Sample_1   1  22838492   G   C  22838492  22838492  ZBTB40
 4  Sample_1   1  27097733   G   A  27097733  27097733  ARID1A,
 (18408, 8))